Training of mentions extractor:

In [34]:
import mentions_extractor_trainer
import torch
from transformers import AutoModel, AutoTokenizer

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
encoder = AutoModel.from_pretrained("SpanBERT/spanbert-base-cased").to(device)
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')
runner = mentions_extractor_trainer.MentionsExtractorTrainer(encoder, tokenizer, 1024)
runner.train()

Some weights of BertModel were not initialized from the model checkpoint at SpanBERT/spanbert-base-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TypeError: train() missing 2 required positional arguments: 'train_dataloader' and 'val_dataloader'

Mentions linker improvement:

In [ ]:
import torch
import ffnn

k = 5
emb_size = 2

top_spans_mention_score = torch.rand(k)
top_spans_embeddings = torch.rand([k, emb_size])

# create a matrix containing the antecedents' indices of the top scoring spans
antecedents_distance = torch.tile(torch.arange(0, k) + 1, [k, 1])  # [k ,k]
print("Antecedents indices:\n" + str(antecedents_distance))
print("Antecedents indices size:\n" + str(antecedents_distance.size()))

raw_antecedents = torch.unsqueeze(torch.arange(k), 1) - antecedents_distance  # [k, k]
print("Raw antecedents:\n" + str(raw_antecedents))
print("Raw antecedents size:\n" + str(raw_antecedents.size()))

# create a mask for out of bound indices
antecedents_mask = raw_antecedents >= 0  # [k, k]
print("Raw antecedents mask:\n" + str(antecedents_mask))
print("Raw antecedents size:\n" + str(antecedents_mask.size()))

antecedents = torch.maximum(raw_antecedents, torch.tensor(0))  # [k, k]
print("Antecedents:\n" + str(antecedents))
print("Antecedents size:\n" + str(antecedents.size()))

pairs_mention_score = top_spans_mention_score.unsqueeze(1) + top_spans_mention_score[antecedents]  # [k, k]
# use mask to put out of bound values to -inf
pairs_mention_score += torch.log(antecedents_mask.float())  # [k, k]
print("Pairs mention score:\n" + str(pairs_mention_score))
print("Pairs mention score size:\n" + str(pairs_mention_score.size()))

antecedents_emb = top_spans_embeddings[antecedents]  # [k, k, emb]
print("Antecendent emb size:\n" + str(antecedents_emb.size()))

target_embeddings = top_spans_embeddings.unsqueeze(1).tile([1, k, 1])  # [k, 1, emb]
print("Target emb size:\n" + str(target_embeddings.size()))
mult_embeddings = target_embeddings * top_spans_embeddings  # [k, k, emb]
print("Multiplication emb size:\n" + str(mult_embeddings.size()))
pairs_embeddings = torch.cat([target_embeddings, antecedents_emb, mult_embeddings], 2)  # [k, k, 3*emb]
print("Pairs emb size:\n" + str(pairs_embeddings.size()))

ffnn = ffnn.FFNN(3 * emb_size)
pairs_antecedent_score = ffnn(pairs_embeddings).squeeze(2)  # [k, k]
print("Pairs antecedent score size: " + str(pairs_antecedent_score.size()))

pairs_score = pairs_mention_score + pairs_antecedent_score


Forward pass in linker:

In [1]:
import torch
import utils
from transformers import AutoModel, AutoTokenizer
import mentions_extractor
import mentions_linker

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
encoder = AutoModel.from_pretrained("SpanBERT/spanbert-base-cased").to(device)
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')
sampleText = "Hello, my name is Damien. It's a cool name I think!"
tokenized_text = tokenizer.encode_plus(sampleText, return_tensors='pt')
tokenized_text["input_ids"] = tokenized_text["input_ids"].squeeze()
tokenized_text["token_type_ids"] = tokenized_text["token_type_ids"].squeeze()
tokenized_text["attention_mask"] = tokenized_text["attention_mask"].squeeze()
encoded_text = utils.spanbert_encode(tokenized_text, encoder)

extractor = mentions_extractor.MentionsExtractor(768, 0.25, 10).to(device)
linker = mentions_linker.MentionsLinker(extractor, 768).to(device)

res = linker(encoded_text)
print(res)


Some weights of BertModel were not initialized from the model checkpoint at SpanBERT/spanbert-base-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


(tensor([ 8, 11, 11, 11], device='cuda:0'), tensor([17, 13, 17, 15], device='cuda:0'), tensor([[  -inf,   -inf,   -inf,   -inf, 0.0000],
        [0.0956,   -inf,   -inf,   -inf, 0.0000],
        [0.0905, 0.0893,   -inf,   -inf, 0.0000],
        [0.0995, 0.0926, 0.0957,   -inf, 0.0000]], device='cuda:0',
       grad_fn=<CatBackward0>), tensor([[0, 0, 0, 0],
        [0, 0, 0, 0],
        [1, 0, 0, 0],
        [2, 1, 0, 0]], device='cuda:0'), tensor([[False, False, False, False],
        [ True, False, False, False],
        [ True,  True, False, False],
        [ True,  True,  True, False]], device='cuda:0'))


Get labels version 1:

In [ ]:
import torch

cand_start = torch.tensor([0, 1, 2, 3, 4, 5]).to(device)
cand_end = torch.tensor([1, 2, 3, 5, 7, 6]).to(device)
true_start = torch.tensor([0, 0, 1, 3, 5]).to(device)
true_end = torch.tensor([0, 1, 3, 5, 6]).to(device)
labels = torch.tensor([2, 1, 2, 1, 3]).to(device)

same_start = torch.eq(torch.unsqueeze(true_start, 1), torch.unsqueeze(cand_start, 0))
same_end = torch.eq(torch.unsqueeze(true_end, 1), torch.unsqueeze(cand_end, 0))
same_span = torch.logical_and(same_start, same_end)
print(same_span)
candidate_labels = torch.matmul(torch.unsqueeze(labels, 0).float(), same_span.float())
print(candidate_labels)
print("\nAlternative:\n")


Get labels version 2:

In [24]:
import torch

#cand_start = torch.tensor([0, 1, 2, 3, 4, 5])
#cand_end = torch.tensor([1, 2, 3, 5, 7, 6])
cand_start = torch.tensor([0, 0, 1, 3, 5, 0])
cand_end = torch.tensor([0, 1, 3, 5, 6, 0])
true_start = torch.tensor([0, 0, 1, 3, 5])
true_end = torch.tensor([0, 1, 3, 5, 6])
labels = torch.tensor([2, 1, 2, 1, 1])
antecedents = torch.tensor([[0, 0, 0, 0, 0, 0],
                            [0, 0, 0, 0, 0, 0],
                            [1, 0, 0, 0, 0, 0],
                            [2, 1, 0, 0, 0, 0],
                            [3, 2, 1, 0, 0, 0],
                            [4, 3, 2, 1, 0, 0]])
antecedents_mask = torch.tensor([[False, False, False, False, False, False],
                                 [True, False, False, False, False, False],
                                 [True, True, False, False, False, False],
                                 [True, True, True, False, False, False],
                                 [True, True, True, True, False, False],
                                 [True, True, True, True, True, False]])

pairs_score = torch.rand([6, 6]) + torch.log(antecedents_mask)
print(pairs_score)

cand_indices = torch.stack([cand_start, cand_end], dim=1)
true_indices = torch.stack([true_start, true_end], dim=1)

predicted_expand = cand_indices.unsqueeze(1).expand(-1, true_indices.shape[0], cand_indices.shape[1])
print(predicted_expand.size())
true_expand = true_indices.unsqueeze(0).expand(cand_indices.shape[0], -1, cand_indices.shape[1])
print(true_expand.size())

same_span = (predicted_expand == true_expand).all(-1)
print(same_span.size())
candidate_labels = torch.matmul(same_span.float(), torch.unsqueeze(labels, 1).float()).squeeze().long()

print(candidate_labels)

antecedents_cluster = candidate_labels[antecedents] + torch.log(antecedents_mask.float())

print(antecedents_cluster)

antecedent_same_cluster = torch.logical_and(torch.eq(antecedents_cluster, candidate_labels.unsqueeze(1)), antecedents_cluster > 0)

print(antecedent_same_cluster)

has_dummy_antecedent = torch.logical_not(antecedent_same_cluster.any(dim=1, keepdim=True))

antecedents_labels = torch.cat([antecedent_same_cluster, has_dummy_antecedent], 1).float()

all_pairs_score = torch.cat([pairs_score, torch.zeros([6, 1])], 1)
true_pairs_score = all_pairs_score + torch.log(antecedents_labels)
print(antecedents_labels)
print(all_pairs_score)
print(true_pairs_score)

log_sum_true_antecedents_score = torch.logsumexp(true_pairs_score, 1)  # [k]
log_sum_all_antecedents_score = torch.logsumexp(all_pairs_score, 1)  # [k]
print(log_sum_true_antecedents_score)
print(log_sum_all_antecedents_score)
loss = -torch.sum(log_sum_true_antecedents_score - log_sum_all_antecedents_score)
print(loss)
print(loss.size())










tensor([[  -inf,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.0960,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.0356, 0.4923,   -inf,   -inf,   -inf,   -inf],
        [0.8172, 0.2301, 0.0771,   -inf,   -inf,   -inf],
        [0.0318, 0.8997, 0.1313, 0.3007,   -inf,   -inf],
        [0.5038, 0.8798, 0.8353, 0.9409, 0.8266,   -inf]])
torch.Size([6, 5, 2])
torch.Size([6, 5, 2])
torch.Size([6, 5])
tensor([2, 1, 2, 1, 1, 2])
tensor([[-inf, -inf, -inf, -inf, -inf, -inf],
        [2., -inf, -inf, -inf, -inf, -inf],
        [1., 2., -inf, -inf, -inf, -inf],
        [2., 1., 2., -inf, -inf, -inf],
        [1., 2., 1., 2., -inf, -inf],
        [1., 1., 2., 1., 2., -inf]])
tensor([[False, False, False, False, False, False],
        [False, False, False, False, False, False],
        [False,  True, False, False, False, False],
        [False,  True, False, False, False, False],
        [ True, False,  True, False, False, False],
        [False, False,  True, False,  True, False]])
t

In [39]:
import torch

t = torch.tensor([[1, 2, 3],
                  [2, 4, 5],
                  [1, 1, 3],
                  [2, 2, 2]])
t2 = t[:, 2]
print(t2.size())
t3 = t[:, :2]
print(t3.size())

torch.Size([4])
torch.Size([4, 2])
